# 07 — Decorators

A decorator is a function that takes a function and returns a
replacement. That is the whole idea — everything else is applying it
carefully: preserving metadata, handling arguments, stacking, decorating
methods and classes, and keeping the wrapper introspectable.

`#decorators` `#closures` `#functools` `#metaprogramming`

<sub>Setup cell — run it once per session so `check()` works.</sub>

In [ ]:
# Wire the grader up. Run this cell first, every session.
import pathlib
import sys

_here = pathlib.Path.cwd()
_root = next((p for p in (_here, *_here.parents)
              if (p / "tutorial" / "grader.py").exists()), None)
if _root is None:
    raise RuntimeError("run this notebook from inside the tutorial repository")
sys.path.insert(0, str(_root))

from tutorial.grader import check, hint, solution   # noqa: E402

# A notebook kernel already owns an event loop, so asyncio.run() would refuse
# to start another one. Hand those coroutines a private loop on a worker
# thread instead, so the asyncio lessons run here exactly as they do in a
# script. (Outside a notebook this wrapper is a no-op.)
import asyncio as _asyncio                                    # noqa: E402
from concurrent.futures import ThreadPoolExecutor as _Pool    # noqa: E402

_plain_run = _asyncio.run


def _run_anywhere(coro, **kwargs):
    try:
        _asyncio.get_running_loop()
    except RuntimeError:
        return _plain_run(coro, **kwargs)
    with _Pool(1) as pool:
        return pool.submit(_plain_run, coro, **kwargs).result()


_asyncio.run = _run_anywhere

print("grader ready — Python", sys.version.split()[0])

## Desugaring

```python
@decorate
def f(): ...
```

is exactly

```python
def f(): ...
f = decorate(f)
```

The decorator runs **once**, at definition time. Whatever it returns
is bound to the name — it need not even be callable, though it almost
always is.

In [ ]:
def announce(fn):
    print(f"  [decorating {fn.__name__} at import time]")
    def wrapper(*args, **kwargs):
        print(f"  [calling {fn.__name__}]")
        return fn(*args, **kwargs)
    return wrapper

@announce
def work(x):
    return x * 2

print("definition finished")
print("result:", work(21))
print("what the name points at now:", work.__name__)

Note the last line: the name `work` now refers to `wrapper`. That
broken `__name__` is why `functools.wraps` exists.

## `functools.wraps` — never skip it

`@wraps(fn)` copies `__name__`, `__doc__`, `__qualname__`,
`__module__`, `__dict__` and `__annotations__` onto the wrapper, and
sets `__wrapped__ = fn`. That last one is what lets
`inspect.signature` see through the wrapper.

In [ ]:
import functools
import inspect

def bare(fn):
    def wrapper(*a, **k): return fn(*a, **k)
    return wrapper

def proper(fn):
    @functools.wraps(fn)
    def wrapper(*a, **k): return fn(*a, **k)
    return wrapper

def target(x: int, y: int = 2) -> int:
    '''Multiply two numbers.'''
    return x * y

b, p = bare(target), proper(target)
print(f"{'':10} {'__name__':10} {'signature':22} doc")
for label, fn in (("bare", b), ("wraps", p)):
    print(f"{label:10} {fn.__name__:10} {str(inspect.signature(fn)):22} {fn.__doc__}")
print("__wrapped__ points home:", p.__wrapped__ is target)

### 🧠 Quiz

Which does `functools.wraps` NOT do?

- **a)** Copy `__name__` and `__doc__`
- **b)** Set `__wrapped__` on the wrapper
- **c)** Make the wrapper accept the same arguments as the original
- **d)** Copy `__annotations__`

<details>
<summary><b>answer</b></summary>

**c** — It only copies metadata. The wrapper's real signature is still `(*args, **kwargs)` — `inspect.signature` merely *reports* the original by following `__wrapped__`.

</details>

## Decorators with arguments: one more layer

`@deco(x)` means `f = deco(x)(f)`. So a parameterised decorator is a
function returning a decorator returning a wrapper — three levels.

In [ ]:
import functools
import time

def retry(times=3, delay=0.0, exceptions=(Exception,)):
    def decorate(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last = None
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as exc:
                    last = exc
                    print(f"  attempt {attempt} failed: {exc}")
                    if attempt < times:
                        time.sleep(delay)
            raise last
        return wrapper
    return decorate

calls = []

@retry(times=3)
def flaky():
    calls.append(1)
    if len(calls) < 3:
        raise ConnectionError("network hiccup")
    return "succeeded on attempt " + str(len(calls))

print(flaky())

## Supporting both `@deco` and `@deco(...)`

Users will inevitably write `@deco` without parentheses. Detect it:
if the decorator is called with a single positional argument that is
callable and no keywords, it was used bare.

In [ ]:
import functools

def logged(fn=None, *, prefix="LOG"):
    if fn is None:                       # called as @logged(prefix=...)
        return functools.partial(logged, prefix=prefix)

    @functools.wraps(fn)                 # called as @logged
    def wrapper(*args, **kwargs):
        print(f"  [{prefix}] {fn.__name__}{args}")
        return fn(*args, **kwargs)
    return wrapper

@logged
def bare(x): return x

@logged(prefix="TRACE")
def parameterised(x): return x

bare(1)
parameterised(2)

The keyword-only `*` is what makes this safe: `@logged` passes the
function positionally, and `@logged(prefix=...)` can only pass
keywords, so the two cases can never be confused.

### ❓ Predict the output

Read the cell below and decide what it prints **before** running it. Then run it.

In [ ]:
def tag(name):
    def decorate(fn):
        def wrapper(*a, **k):
            return f"<{name}>{fn(*a, **k)}</{name}>"
        return wrapper
    return decorate

@tag("b")
@tag("i")
def text():
    return "hi"

print(text())

<details>
<summary><b>the answer, and why</b></summary>

```
<b><i>hi</i></b>
```

Stacked decorators apply **bottom-up**: `text = tag("b")(tag("i")(text))`.
So `i` wraps the function first and ends up innermost, while `b`
wraps that and ends up outermost.

</details>

### 🧠 Quiz

In what order do stacked decorators apply?

- **a)** Top to bottom; the top one is innermost
- **b)** Bottom to top; the bottom one is innermost
- **c)** Alphabetically by decorator name
- **d)** All at once, in an undefined order

<details>
<summary><b>answer</b></summary>

**b** — The one closest to `def` runs first and is therefore innermost; the topmost decorator wraps everything and runs first at call time.

</details>

## Decorating methods

A decorator sees the plain function *before* it becomes a bound
method, so `self` simply arrives as the first positional argument.
The subtleties are with `classmethod`/`staticmethod`, which must be
the **outermost** decorators, since they return descriptors, not
functions.

In [ ]:
import functools

def trace(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        where = type(args[0]).__name__ + "." if args else ""
        print(f"  -> {where}{fn.__name__}")
        return fn(*args, **kwargs)
    return wrapper

class Service:
    @trace
    def fetch(self, key):
        return f"value-for-{key}"

    @classmethod                 # OUTERMOST
    @trace
    def build(cls, name):
        return cls()

    @staticmethod
    @trace
    def helper(x):
        return x + 1

    @property
    @trace
    def status(self):
        return "ok"

s = Service()
print(s.fetch("a"))
print(Service.build("svc"))
print(Service.helper(1))
print(s.status)

Put `@classmethod` *below* `@trace` and you would be decorating a
classmethod object — not callable — and get a `TypeError` at call
time. Rule of thumb: descriptor-producing decorators go on top.

## Class-based decorators, and decorating classes

- A **class as decorator** — implement `__call__`. Useful when the
  decorator needs rich state or its own methods.
- A **decorator applied to a class** — receives and returns a class.
  `dataclass`, `total_ordering`, and registration decorators work
  this way.

In [ ]:
import functools

class CountCalls:
    def __init__(self, fn):
        functools.update_wrapper(self, fn)     # the wraps equivalent
        self.fn = fn
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.fn(*args, **kwargs)

    def reset(self):
        self.count = 0

@CountCalls
def ping():
    return "pong"

ping(); ping(); ping()
print(f"{ping.__name__} called {ping.count} times")
ping.reset()
print("after reset:", ping.count)

One caveat: a class-based decorator on a *method* breaks binding —
the instance is not passed, because your class is not a descriptor.
Add `__get__` returning `functools.partial(self.__call__, obj)` if
you need that.

In [ ]:
# Decorating a class: a plugin registry.
REGISTRY = {}

def register(name):
    def decorate(cls):
        if name in REGISTRY:
            raise ValueError(f"duplicate plugin {name!r}")
        REGISTRY[name] = cls
        cls.plugin_name = name           # augment the class
        return cls                       # return it unchanged otherwise
    return decorate

@register("json")
class JsonExporter:
    def export(self, data): return "{...}"

@register("csv")
class CsvExporter:
    def export(self, data): return "a,b,c"

print("registry:", REGISTRY)
print("augmented:", JsonExporter.plugin_name)
print("still a normal class:", JsonExporter().export({}))

## The stdlib decorators worth knowing

| decorator | what it does |
|---|---|
| `functools.cache` | unbounded memoisation (3.9+) |
| `functools.lru_cache(maxsize)` | bounded memoisation with stats |
| `functools.cached_property` | compute once per instance, store in `__dict__` |
| `functools.singledispatch` | overload on the first argument's type |
| `functools.total_ordering` | fill in comparison methods |
| `contextlib.contextmanager` | generator → context manager |
| `atexit.register` | run at interpreter shutdown |

In [ ]:
import functools
import time

@functools.lru_cache(maxsize=128)
def slow_square(n):
    time.sleep(0.01)
    return n * n

start = time.perf_counter()
[slow_square(i % 10) for i in range(50)]
elapsed = time.perf_counter() - start
print(f"50 calls in {elapsed*1000:.0f} ms (only 10 were real)")
print("stats:", slow_square.cache_info())
slow_square.cache_clear()
print("after clear:", slow_square.cache_info())

Caveats: arguments must be **hashable**, the cache is shared across
all callers, and on a method it keeps `self` alive forever — a
classic memory leak. Prefer `cached_property` on instances.

In [ ]:
import functools

@functools.singledispatch
def describe(value):
    return f"some {type(value).__name__}"

@describe.register
def _(value: int):
    return f"the integer {value}"

@describe.register(list)
def _(value):
    return f"a list of {len(value)} items"

@describe.register(str)
def _(value):
    return f"the string {value!r}"

for item in (42, "hi", [1, 2, 3], 3.5):
    print(" ", describe(item))
print("registry:", sorted(c.__name__ for c in describe.registry))

`singledispatch` dispatches on the runtime type of the **first**
argument, honouring the MRO — so registering `Sequence` catches
tuples and lists both. `singledispatchmethod` does the same for
methods, dispatching on the first argument after `self`.

In [ ]:
import functools

class Dataset:
    def __init__(self, rows): self.rows = rows

    @functools.cached_property
    def summary(self):
        print("  (computing summary -- expensive)")
        return {"n": len(self.rows), "total": sum(self.rows)}

d = Dataset([1, 2, 3])
print(d.summary)
print(d.summary, "  <- second access, no recompute")
print("stored in the instance dict:", "summary" in d.__dict__)
del d.__dict__["summary"]         # this is how you invalidate it
print(d.summary)

`cached_property` needs a writable `__dict__`, so it cannot be
used with `__slots__`. It is also **not** thread-safe as of 3.12:
two threads may both compute the value.

---

## ⚒ Exercise — A production-grade retry decorator  
`07-decorators/retry` · difficulty: **tricky**

Write `retry(attempts=3, exceptions=(Exception,), backoff=None, on_retry=None)`.

- retries the wrapped call up to `attempts` times total
- only catches the listed `exceptions`; anything else propagates
  immediately
- if every attempt fails, re-raise the **last** exception
- `backoff(attempt) -> float` is called between attempts with the
  1-based attempt number that just failed; the returned number is
  passed to a `sleep` you must call via the module-level name
  `sleep` so tests can replace it
- `on_retry(exc, attempt)` is called after each failed attempt
  that will be retried (not after the final one)
- `attempts < 1` raises `ValueError` at decoration time
- metadata is preserved, and the wrapper exposes `.attempts`
- a successful first call must not sleep at all

Edit the next cell, run it, then run the check cell below it.

In [ ]:
import functools
import time

sleep = time.sleep     # tests monkeypatch this name


def retry(attempts=3, exceptions=(Exception,), backoff=None, on_retry=None):
    ...

In [ ]:
check("07-decorators/retry")

<details>
<summary><b>hints</b></summary>

1. Call the sleep through the module global — `sleep(...)`, not `time.sleep(...)` — so a test can swap it out.
2. The loop shape is `for attempt in range(1, attempts + 1)` with the `raise` after the loop, or inside `if attempt == attempts`.
3. Only sleep when another attempt will actually happen.
4. `raise` with no argument inside `except` re-raises the current exception with its original traceback.

</details>

<details>
<summary><b>reference solution</b></summary>

```python
import functools
import time

sleep = time.sleep

def retry(attempts=3, exceptions=(Exception,), backoff=None, on_retry=None):
    if attempts < 1:
        raise ValueError("attempts must be at least 1")

    def decorate(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as exc:
                    if attempt == attempts:
                        raise
                    if on_retry is not None:
                        on_retry(exc, attempt)
                    if backoff is not None:
                        sleep(backoff(attempt))
        wrapper.attempts = attempts
        return wrapper
    return decorate
```

</details>

---

## ⚒ Exercise — A memoiser with TTL, key policy, and stats  
`07-decorators/memoize` · difficulty: **brutal**

Write `memoize(ttl=None, key=None, maxsize=None)` — a caching
decorator you would actually ship.

- caches results keyed by the call arguments
- `key(*args, **kwargs)` overrides key construction when given
- `f(1)` and `f(x=1)` must hit the **same** cache entry by default
  (normalise the call with `inspect.signature`)
- `ttl` in seconds; expired entries recompute. Read the clock via
  the module-level name `now` so tests can control time
- `maxsize` evicts the **least recently used** entry when exceeded
- `f.cache_info()` → dict with `hits`, `misses`, `size`
- `f.cache_clear()` empties the cache and zeroes the counters
- unhashable arguments must not crash: fall back to computing
  without caching, and count that as a miss

Edit the next cell, run it, then run the check cell below it.

In [ ]:
import functools
import inspect
import time
from collections import OrderedDict

now = time.monotonic     # tests monkeypatch this name


def memoize(ttl=None, key=None, maxsize=None):
    ...

In [ ]:
check("07-decorators/memoize")

<details>
<summary><b>hints</b></summary>

1. `sig.bind(*args, **kwargs)` + `apply_defaults()` then a tuple of `sorted(bound.arguments.items())` gives a call-style-independent key.
2. `OrderedDict.move_to_end(k)` on a hit, and `popitem(last=False)` to evict, gives you LRU in two lines.
3. Store `(value, stored_at)` so the TTL check is `now() - stored_at > ttl`.
4. Wrap the key construction in `try/except TypeError` to catch unhashable arguments — and remember to still call the function.

</details>

<details>
<summary><b>reference solution</b></summary>

```python
import functools
import inspect
import time
from collections import OrderedDict

now = time.monotonic

def memoize(ttl=None, key=None, maxsize=None):
    def decorate(fn):
        sig = inspect.signature(fn)
        cache = OrderedDict()
        stats = {"hits": 0, "misses": 0}

        def make_key(args, kwargs):
            if key is not None:
                return key(*args, **kwargs)
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            return tuple(sorted(bound.arguments.items()))

        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            try:
                cache_key = make_key(args, kwargs)
                hash(cache_key)
            except TypeError:
                stats["misses"] += 1
                return fn(*args, **kwargs)

            if cache_key in cache:
                value, stored_at = cache[cache_key]
                if ttl is None or now() - stored_at <= ttl:
                    cache.move_to_end(cache_key)
                    stats["hits"] += 1
                    return value
                del cache[cache_key]

            stats["misses"] += 1
            value = fn(*args, **kwargs)
            cache[cache_key] = (value, now())
            cache.move_to_end(cache_key)
            if maxsize is not None and len(cache) > maxsize:
                cache.popitem(last=False)
            return value

        wrapper.cache_info = lambda: {**stats, "size": len(cache)}

        def cache_clear():
            cache.clear()
            stats.update(hits=0, misses=0)
        wrapper.cache_clear = cache_clear
        return wrapper
    return decorate
```

</details>

## Takeaways

- `@d` is `f = d(f)`, run once at definition time.
- Always `functools.wraps` — otherwise you destroy introspection.
- Parameterised decorators are three nested functions; keyword-only
  parameters let one decorator support both call styles.
- Stacking applies bottom-up; `classmethod`/`staticmethod` go on top.
- Reach for `cache`, `cached_property`, `singledispatch` before
  writing your own.

---

### ✅ Lesson complete

Exercises in this lesson: **2**. Check your standing across the whole course with:

```
uv run learn.py list
```